# Support Integrity Auditor (Part 1)
This notebook loads the CRM dataset, cleans data, engineers features, generates pseudo-labels, and prepares training data.


In [1]:

import pandas as pd
import numpy as np

# Update with your dataset path
DATA_PATH = "customer_support_tickets.csv"

df = pd.read_csv(DATA_PATH)
df.head(8)


,Ticket_ID,Customer_Name,Customer_Email,Ticket_Subject,Ticket_Description,Issue_Category,Priority_Level,Ticket_Channel,Submission_Date,Resolution_Time_Hours,Assigned_Agent,Satisfaction_Score
0,TKT-100000,George Simon,lisastrickland@example.com,Hours of operation - Individual,"Hi Support, Where is your headquarters located...",General Inquiry,High,Web Form,2025-07-02,43,David Kim,5
1,TKT-100001,Scott Thompson,wevans@example.org,Data not syncing - Card,"Hi Support, The application crashes every time...",Technical,High,Chat,2025-06-28,41,Elena Rodriguez,5
2,TKT-100002,Jennifer Smith,oleonard@example.net,2FA issues - Question,"Hi Support, How do I upgrade to the Enterprise...",Account,High,Web Form,2025-02-05,7,Anya Sharma,5
3,TKT-100003,Rachel Bullock,katherine67@example.net,Login failed - Let,"Hi Support, The dashboard is not loading any d...",Technical,Low,Web Form,2025-03-20,41,Anya Sharma,5
4,TKT-100004,Thomas Parks DDS,raykelsey@example.com,Refund status - Attention,"Hi Support, I have been trying to update my pa...",Billing,Medium,Email,2025-04-27,40,David Kim,5
5,TKT-100005,Ashley Stewart,gibsonrose@example.com,Office location - National,"Hi Support, Where is your headquarters located...",General Inquiry,Medium,Chat,2025-10-02,23,Elena Rodriguez,4
6,TKT-100006,Ashley Bennett,gfox@example.org,Password reset - Body,"Hi Support, How do I upgrade to the Enterprise...",Account,Medium,Chat,2024-09-28,106,David Kim,3
7,TKT-100007,Travis Blanchard,francismegan@example.net,Payment failed - Win,"Hi Support, My subscription renewed automatica...",Billing,Medium,Email,2024-10-26,27,David Kim,1


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Ticket_ID              20000 non-null  object
 1   Customer_Name          20000 non-null  object
 2   Customer_Email         20000 non-null  object
 3   Ticket_Subject         20000 non-null  object
 4   Ticket_Description     20000 non-null  object
 5   Issue_Category         20000 non-null  object
 6   Priority_Level         20000 non-null  object
 7   Ticket_Channel         20000 non-null  object
 8   Submission_Date        20000 non-null  object
 9   Resolution_Time_Hours  20000 non-null  int64 
 10  Assigned_Agent         20000 non-null  object
 11  Satisfaction_Score     20000 non-null  int64 
dtypes: int64(2), object(10)
memory usage: 1.8+ MB


In [3]:
import re

def remove_last_filler(text):
    sentences = re.split(r'(?<=[.!?])\s+', text)
    if len(sentences) > 1:
        return " ".join(sentences[:-1])
    return text

df["Ticket_Description"] = df["Ticket_Description"].apply(remove_last_filler)

df.head(8)

,Ticket_ID,Customer_Name,Customer_Email,Ticket_Subject,Ticket_Description,Issue_Category,Priority_Level,Ticket_Channel,Submission_Date,Resolution_Time_Hours,Assigned_Agent,Satisfaction_Score
0,TKT-100000,George Simon,lisastrickland@example.com,Hours of operation - Individual,"Hi Support, Where is your headquarters located?",General Inquiry,High,Web Form,2025-07-02,43,David Kim,5
1,TKT-100001,Scott Thompson,wevans@example.org,Data not syncing - Card,"Hi Support, The application crashes every time...",Technical,High,Chat,2025-06-28,41,Elena Rodriguez,5
2,TKT-100002,Jennifer Smith,oleonard@example.net,2FA issues - Question,"Hi Support, How do I upgrade to the Enterprise...",Account,High,Web Form,2025-02-05,7,Anya Sharma,5
3,TKT-100003,Rachel Bullock,katherine67@example.net,Login failed - Let,"Hi Support, The dashboard is not loading any d...",Technical,Low,Web Form,2025-03-20,41,Anya Sharma,5
4,TKT-100004,Thomas Parks DDS,raykelsey@example.com,Refund status - Attention,"Hi Support, I have been trying to update my pa...",Billing,Medium,Email,2025-04-27,40,David Kim,5
5,TKT-100005,Ashley Stewart,gibsonrose@example.com,Office location - National,"Hi Support, Where is your headquarters located?",General Inquiry,Medium,Chat,2025-10-02,23,Elena Rodriguez,4
6,TKT-100006,Ashley Bennett,gfox@example.org,Password reset - Body,"Hi Support, How do I upgrade to the Enterprise...",Account,Medium,Chat,2024-09-28,106,David Kim,3
7,TKT-100007,Travis Blanchard,francismegan@example.net,Payment failed - Win,"Hi Support, My subscription renewed automatica...",Billing,Medium,Email,2024-10-26,27,David Kim,1


In [4]:

# Basic preprocessing
text_cols = ["Ticket_Subject","Ticket_Description"]
for c in text_cols:
    df[c] = df[c].fillna("").astype(str)

df["combined_text"] = (
    df["Ticket_Subject"] + " " + df["Ticket_Description"]
).str.lower()

df["Resolution_Time_Hours"] = pd.to_numeric(
    df["Resolution_Time_Hours"], errors="coerce"
)
df["Resolution_Time_Hours"] = df["Resolution_Time_Hours"].fillna(
    df["Resolution_Time_Hours"].median()
)

df.head()


,Ticket_ID,Customer_Name,Customer_Email,Ticket_Subject,Ticket_Description,Issue_Category,Priority_Level,Ticket_Channel,Submission_Date,Resolution_Time_Hours,Assigned_Agent,Satisfaction_Score,combined_text
0,TKT-100000,George Simon,lisastrickland@example.com,Hours of operation - Individual,"Hi Support, Where is your headquarters located?",General Inquiry,High,Web Form,2025-07-02,43,David Kim,5,"hours of operation - individual hi support, wh..."
1,TKT-100001,Scott Thompson,wevans@example.org,Data not syncing - Card,"Hi Support, The application crashes every time...",Technical,High,Chat,2025-06-28,41,Elena Rodriguez,5,"data not syncing - card hi support, the applic..."
2,TKT-100002,Jennifer Smith,oleonard@example.net,2FA issues - Question,"Hi Support, How do I upgrade to the Enterprise...",Account,High,Web Form,2025-02-05,7,Anya Sharma,5,"2fa issues - question hi support, how do i upg..."
3,TKT-100003,Rachel Bullock,katherine67@example.net,Login failed - Let,"Hi Support, The dashboard is not loading any d...",Technical,Low,Web Form,2025-03-20,41,Anya Sharma,5,"login failed - let hi support, the dashboard i..."
4,TKT-100004,Thomas Parks DDS,raykelsey@example.com,Refund status - Attention,"Hi Support, I have been trying to update my pa...",Billing,Medium,Email,2025-04-27,40,David Kim,5,"refund status - attention hi support, i have b..."


## Rule-based urgency scoring


In [5]:

urgent_words = [
    "urgent","critical","outage","down","security",
    "payment","failed","cannot","blocked","breach","crash","suspicious"
]

def keyword_score(text):
    score = 0
    for w in urgent_words:
        if w in text:
            score += 1
    return score

df["keyword_score"] = df["combined_text"].apply(keyword_score)
df[["combined_text","keyword_score"]].head()


,combined_text,keyword_score
0,"hours of operation - individual hi support, wh...",0
1,"data not syncing - card hi support, the applic...",1
2,"2fa issues - question hi support, how do i upg...",0
3,"login failed - let hi support, the dashboard i...",1
4,"refund status - attention hi support, i have b...",1


In [6]:
df[["combined_text","keyword_score","Priority_Level"]][df["Priority_Level"]=="High"].head()

,combined_text,keyword_score,Priority_Level
0,"hours of operation - individual hi support, wh...",0,High
1,"data not syncing - card hi support, the applic...",1,High
2,"2fa issues - question hi support, how do i upg...",0,High
8,"login failed - son hi support, i cannot log in...",2,High
36,"phishing attempt - close hi support, i receive...",1,High


In [7]:
from src.llm_severity import get_llm_severity

severity = get_llm_severity(df["Ticket_Subject"][0],df["Ticket_Description"][0])

severity

ModuleNotFoundError: No module named 'transformers'

## Pseudo severity fusion


In [ ]:

# res_scaled = (
#     (df["Resolution Time"] - df["Resolution Time"].min())
#     /(df["Resolution Time"].max()-df["Resolution Time"].min())
# )

# fusion = 0.7*(df["keyword_score"]/max(df["keyword_score"].max(),1)) + 0.3*res_scaled

# def sev(x):
#     if x < 0.25:
#         return "Low"
#     if x < 0.5:
#         return "Medium"
#     if x < 0.75:
#         return "High"
#     return "Critical"

# df["inferred_severity"] = fusion.apply(sev)

# priority_map = {"Low":0,"Medium":1,"High":2,"Critical":3}
# df["pseudo_label"] = (
#     df["inferred_severity"].map(priority_map)
#     != df["Ticket Priority"].map(priority_map)
# ).astype(int)

# df[["Ticket Priority","inferred_severity","pseudo_label"]].head()


Subsequent parts should replace the heuristic with sentence-transformers and a fine-tuned DeBERTa model.
